# Этап 13: SmoothQuant (Dual-Track) — Укрощение выбросов

**SmoothQuant** (arXiv:2211.10438) переносит сложность квантования с активаций на веса. 
Это критически важно для Llama, где выбросы (outliers) могут быть в 100 раз больше средних значений.

### Идея:
$Y = (X diag(s)^{-1}) \cdot (diag(s) W)$

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from src.model import GPTLanguageModel, device, get_batch

# 1. Загружаем модели
nanogpt = GPTLanguageModel().to(device)
nanogpt.load_state_dict(torch.load('model_ckpt.pt', map_location=device))

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
llama = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_id)

def smooth_quant_layer(layer, act_max, alpha=0.5):
    with torch.no_grad():
        # act_max: максимум активаций по каждому каналу
        weight_max = layer.weight.abs().max(dim=0)[0].float()
        s = act_max.pow(alpha) / (weight_max.pow(1-alpha) + 1e-6)
        
        # Масштабируем веса (умножаем столбцы)
        layer.weight.data *= s.view(1, -1).to(layer.weight.dtype)
        return s

# 2. nanoGPT: Сглаживаем первый слой MLP
xb, _ = get_batch('val') # Проп: (B, T, C)
nano_acts_max = torch.randn(384).to(device).abs() * 10.0 # Симуляция активаций
nano_layer = nanogpt.blocks[0].ffwd.net[0]
s_nano = smooth_quant_layer(nano_layer, nano_acts_max)
print(f"nanoGPT: SmoothQuant applied. Max scale: {s_nano.max().item():.2f}")

# 3. Llama: Сглаживаем gate_proj
# В Llama выбросы в активациях достигают огромных масштабов
llama_acts_max = torch.randn(2048).to(device).abs().half() * 50.0 # Симуляция выбросов Llama
llama_layer = llama.model.layers[0].mlp.gate_proj
s_llama = smooth_quant_layer(llama_layer, llama_acts_max)
print(f"Llama: SmoothQuant applied. Max scale: {s_llama.max().item():.2f}")